In [0]:
import requests
import pandas as pd
import pyspark.sql.functions as sf

# Reads the cities from the csv file containing the cities and coordinates
cities_git_url = "https://raw.githubusercontent.com/samkuzmo/Weather-Data-Pipeline/main/Data/cidades.csv"

cidades_response = requests.get(cities_git_url)
cities_df = pd.read_csv(pd.io.common.StringIO(cidades_response.text))
dim_cidades = spark.createDataFrame(cities_df)

# Openweather API Key reading from Databricks Secrets
api_key = dbutils.secrets.get(scope = "weather", key = "openweather_api_key")

cities = dim_cidades.collect()
results = []

# API calls loop for weather data from each city
for city in cities:
    lat = float(city["lat"])
    lon = float(city["lon"])
    name = city["city"]
    
    url = "https://api.openweathermap.org/data/2.5/weather"
    
    params = {
        "lat": city["lat"],
        "lon": city["lon"],
        "units": "metric",
        "appid": api_key
    }

    response = requests.get(url, params = params)

    if response.status_code == 200:
        data = response.json()
        data['city'] = name
        results.append(data)
    else:
        print(f"Error in {name}: {response.status_code}")

# Pandas DataFrame to Spark DataFrame conversion
pdf = pd.json_normalize(results)
df_bronze = spark.createDataFrame(pdf)

display(df_bronze)

df_bronze.write.format("Delta").mode("overwrite").save("/Volumes/weather/default/01_bronze")